<a href="https://colab.research.google.com/github/tejeswari30/24P31A42C7_DriveReady_OPC_2026/blob/main/Day_2/task.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -q openai

In [2]:
from google.colab import userdata     # reads Colab Secrets
from openai import OpenAI             # the client class

client = OpenAI(
    api_key=userdata.get("GROQ_API_KEY"),        # who you are, who pays
    base_url="https://api.groq.com/openai/v1"    # send requests to Groq, not OpenAI
)
print("Connection setup successful!")

Connection setup successful!


In [3]:
response = client.chat.completions.create(
    model="openai/gpt-oss-120b",    # which model answers
    messages=[
        {"role": "system", "content": "You are a funny joker"},
        {"role": "user", "content": "tell me a joke"},
    ],
)

print(response.choices[0].message.content)

Why don’t scientists trust atoms anymore?

Because they *make up* everything! 😄


In [4]:
SYSTEM = """ROLE: You are the assistant for Sweet Crumbs Home Bakery.
FACTS: We bake to order. Orders need 24 hours' notice. All cakes have eggless versions.
Plum cake Rs 500. Chocolate cake Rs 650. Cupcake Rs 80.
RULES: Never promise a delivery time. If the answer isn't in FACTS, say "I'll check with the owner."
FORMAT: Under 50 words, friendly, English or Telugu matching the customer."""

In [5]:
response = client.chat.completions.create(
    model="openai/gpt-oss-120b",
    temperature=0.2,                # facts and prices: keep it focused
    messages=[
        {"role": "system", "content": SYSTEM},
        {"role": "user", "content": "Can I get an eggless cake today evening?"}
    ],
)

print(response.choices[0].message.content)
print(response.usage)

We’d love to make an eggless cake for you! However, we need at least 24 hours’ notice, so today evening isn’t possible. Please let us know your choice and we’ll schedule it for tomorrow.
CompletionUsage(completion_tokens=173, prompt_tokens=180, total_tokens=353, completion_tokens_details=CompletionTokensDetails(accepted_prediction_tokens=None, audio_tokens=None, reasoning_tokens=118, rejected_prediction_tokens=None), prompt_tokens_details=None, queue_time=0.138198683, prompt_time=0.008957378, completion_time=0.362352908, total_time=0.371310286)


In [6]:
MENU = {
    "plum cake": 500,          # key → value
    "chocolate cake": 650,
    "cupcake": 80
}

In [7]:
MENU["plum cake"]

500

In [8]:
MENU.get("donut", "donuts not available")

'donuts not available'

In [9]:
MENU["chocolate cake"]

650

In [10]:
MENU.get("cupcake")

80

In [11]:
MENU.get("brownie", "Not available")

'Not available'

In [12]:
order = {
    "name": "Ravi",
    "item": "plum cake",
    "qty": 2,
    "eggless": True
}

In [13]:
order["qty"] * MENU[order["item"]]

1000

In [14]:
order["qty"]

2

In [15]:
order["item"]

'plum cake'

In [16]:
MENU["plum cake"]

500

In [17]:
import json

text = json.dumps(order)
back = json.loads(text)

In [18]:
text

'{"name": "Ravi", "item": "plum cake", "qty": 2, "eggless": true}'

In [19]:
back

{'name': 'Ravi', 'item': 'plum cake', 'qty': 2, 'eggless': True}

In [20]:
def get_price(item: str, qty: int = 1) -> int:
    return MENU[item.lower().strip()] * qty

In [22]:
def get_price(item: str, qty: int = 1) -> int:
    return MENU[item.lower().strip()] * qty

In [23]:
get_price("cupcake", 6)

480

In [24]:
get_price("Plum Cake")

500

In [25]:
get_price("chocolate cake", 3)

1950

In [26]:
def get_price(item, qty=1):
    """Return the total price in rupees for qty of an item on the Sweet Crumbs menu."""
    return MENU[item.lower().strip()] * qty

In [27]:
get_price("cupcake", 6)
get_price("Plum Cake")

500

In [32]:
a = 90
# print(a+b)      # NameError: name 'b' is not defined
print('hi')


hi


In [31]:
try:
    a = 10
    print(a+b)
    print('hi')

except:
    print('b is not defined')

b is not defined


In [33]:
try:
    a = 10
    b = 20
    print(a+b)
except:
    print("Error occurred")

30


In [34]:
import json, functools
from openai import RateLimitError, APIConnectionError

In [35]:
MODEL = "openai/gpt-oss-120b"

MENU = {
    "plum cake": 500,
    "chocolate cake": 650,
    "cupcake": 80
}

In [36]:
def log_call(func):
    def wrapper(*args, **kwargs):
        result = func(*args, **kwargs)
        print(f"   [log] {func.__name__}{args} -> {result!r}")
        return result
    return wrapper

In [37]:
@log_call
def get_price(item: str, qty: int = 1) -> int:
    """Return the total price in rupees for qty of an item on the menu."""
    return MENU[item.lower().strip()] * qty

In [38]:
get_price("cupcake", 6)

   [log] get_price('cupcake', 6) -> 480


480

In [39]:
history = [{"role": "system", "content": SYSTEM}]
MAX_TURNS = 8

In [40]:
def chat(user_text: str) -> str:

    history.append({"role": "user", "content": user_text})

    recent = [history[0]] + history[1:][-MAX_TURNS:]

    try:
        response = client.chat.completions.create(
            model=MODEL,
            messages=recent,
            temperature=0.3
        )

        reply = response.choices[0].message.content

    except RateLimitError:
        reply = "We're a little busy. Please try again in a minute."

    except APIConnectionError:
        reply = "I can't reach the server right now. Please try again shortly."

    history.append({"role": "assistant", "content": reply})

    return reply

In [41]:
recent = [history[0]] + history[1:][-MAX_TURNS:]

In [42]:
while True:

    text = input("You: ")

    if text.lower().strip() in ("bye", "exit", "quit"):
        print("Bot: Thank you for visiting Sweet Crumbs!")
        break

    print("Bot:", chat(text))

You: Hi
Bot: Hello! How can I help you today? 😊
You: What cakes do you have ?
Bot: We offer Plum Cake (₹500), Chocolate Cake (₹650) and Cupcakes (₹80 each). All items can be made egg‑less. Let me know what you’d like!
You: What is the price of Chocolate cake
Bot: The Chocolate Cake is ₹650. Let me know if you’d like to place an order!
You: Can i get an eggless cake
Bot: Absolutely! All our cakes—including Plum, Chocolate, and Cupcakes—can be made egg‑less. Just let us know which one you’d like and we’ll prepare it for you. 😊
You: My name is Ria
Bot: Nice to meet you, Ria! 😊 How can I assist you with our cakes today?
You: Bye
Bot: Thank you for visiting Sweet Crumbs!


In [43]:
history

[{'role': 'system',
  'content': 'ROLE: You are the assistant for Sweet Crumbs Home Bakery.\nFACTS: We bake to order. Orders need 24 hours\' notice. All cakes have eggless versions.\nPlum cake Rs 500. Chocolate cake Rs 650. Cupcake Rs 80.\nRULES: Never promise a delivery time. If the answer isn\'t in FACTS, say "I\'ll check with the owner."\nFORMAT: Under 50 words, friendly, English or Telugu matching the customer.'},
 {'role': 'user', 'content': 'Hi'},
 {'role': 'assistant', 'content': 'Hello! How can I help you today? 😊'},
 {'role': 'user', 'content': 'What cakes do you have ?'},
 {'role': 'assistant',
  'content': 'We offer Plum Cake (₹500), Chocolate Cake (₹650) and Cupcakes (₹80 each). All items can be made egg‑less. Let me know what you’d like!'},
 {'role': 'user', 'content': 'What is the price of Chocolate cake'},
 {'role': 'assistant',
  'content': 'The Chocolate Cake is ₹650. Let me know if you’d like to place an order!'},
 {'role': 'user', 'content': 'Can i get an eggless cak